In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/voz_hsd_train_wordsegment.jsonl
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/VnCoreNLP-1.2.jar
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/models/ner/vi-pretrainedembeddings.xz
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/models/ner/vi-ner.xz
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/models/ner/vi-500brownclusters.xz
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/models/postagger/vi-tagger
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/models/wordsegmenter/wordsegmenter.rdr
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/models/wordsegmenter/vi-vocab
/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/models/dep/vi-dep.xz
/kaggle/input/models/phanhuycng/phobert-base-v2/pyto

In [2]:
MODEL_PATH    = "/kaggle/input/models/phanhuycng/phobert-base-v2/pytorch/default/1/phobert-base-v2"
DATA_PATH     = "/kaggle/input/datasets/phanhuycng/phobert-pretrain-dataset/voz_hsd_train_wordsegment.jsonl"
OUTPUT_DIR    = "/kaggle/working/phobert-continual"

# Training hyperparams
MAX_LENGTH    = 256       # PhoBERT max = 256
MLM_PROB      = 0.15
BATCH_SIZE    = 32        # điều chỉnh nếu OOM
GRAD_ACCUM    = 4         # effective batch = 128
LR            = 5e-5
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.06
WEIGHT_DECAY  = 0.01
SAVE_STRATEGY = "epoch"   # lưu mỗi epoch
SAVE_TOTAL    = 2         # giữ 2 checkpoint gần nhất
FP16          = True      # RTX PRO 6000 hỗ trợ FP16
SEED          = 42

In [3]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)

print(f"Vocab size : {tokenizer.vocab_size:,}")
print(f"Model params: {model.num_parameters():,}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
RobertaForMaskedLM LOAD REPORT from: /kaggle/input/models/phanhuycng/phobert-base-v2/pytorch/default/1/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
pooler.dense.weight       | UNEXPECTED | 
pooler.dense.bias         | UNEXPECTED | 
lm_head.decoder.bias      | MISSING    | 
lm_head.dense.bias        | MISSING    | 
lm_head.dense.weight      | MISSING    | 
lm_head.layer_norm.weight | MISSING    | 
lm_head.layer_norm.bias   | MISSING    | 
lm_head.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

Vocab size : 64,000
Model params: 135,063,809


In [4]:
from datasets import load_dataset

# File jsonl — mỗi dòng phải có field "text"
raw = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"Số mẫu raw: {len(raw):,}")
print("Sample:", raw[0])

Generating train split: 0 examples [00:00, ? examples/s]

Số mẫu raw: 10,747,733
Sample: {'texts': 'nói nhảm vừa thôi cha ! em nào em nấy đều chân dài , vú to , mông tròn , số_đo 3 vòng nảy_nở hết đó nha !'}


In [5]:
# Nếu field không phải "text", đổi TEXT_FIELD cho đúng
TEXT_FIELD = "texts"

def tokenize_fn(batch):
    return tokenizer(
        batch[TEXT_FIELD],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,          # DataCollator sẽ pad động
        return_special_tokens_mask=True,
    )

tokenized = raw.map(
    tokenize_fn,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=raw.column_names,
    desc="Tokenizing",
)
print(f"Số mẫu sau tokenize: {len(tokenized):,}")
print("Columns:", tokenized.column_names)

Tokenizing (num_proc=2):   0%|          | 0/10747733 [00:00<?, ? examples/s]

Số mẫu sau tokenize: 10,747,733
Columns: ['input_ids', 'special_tokens_mask', 'attention_mask']


In [6]:
def group_texts(batch):
    """Nối toàn bộ token rồi cắt thành chunk MAX_LENGTH."""
    concatenated = {k: sum(batch[k], []) for k in batch.keys()}
    total = len(concatenated["input_ids"])
    # Bỏ phần dư cuối
    total = (total // MAX_LENGTH) * MAX_LENGTH
    result = {
        k: [v[i : i + MAX_LENGTH] for i in range(0, total, MAX_LENGTH)]
        for k, v in concatenated.items()
    }
    return result

grouped = tokenized.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=2,
    desc="Grouping into chunks",
)
print(f"Số chunks sau grouping: {len(grouped):,}")

Grouping into chunks (num_proc=2):   0%|          | 0/10747733 [00:00<?, ? examples/s]

Process ForkPoolWorker-5:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/multiprocess/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/lib/python3.12/dist-packages/multiprocess/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/multiprocess/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/utils/py_utils.py", line 585, in _write_generator_to_queue
    for i, result in enumerate(func(**kwargs)):
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py", line 3997, in _map_single
    for i, batch in iter_outputs(shard_iterable):
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py", line 3950, in iter_outputs
  

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/datasets/utils/py_utils.py", line 610, in iflatmap_unordered
    yield queue.get(timeout=0.05)
          ^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in get
  File "/usr/local/lib/python3.12/dist-packages/multiprocess/managers.py", line 821, in _callmethod
    kind, result = conn.recv()
                   ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/multiprocess/connection.py", line 253, in recv
    buf = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/multiprocess/connection.py", line 433, in _recv_bytes
    buf = self._recv(4)
          ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/multiprocess/connection.py", line 398, in _recv
    chunk = read(handle, remaining)
            ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last)

TypeError: object of type 'NoneType' has no len()

In [7]:
from transformers import DataCollatorForLanguageModeling

split     = tokenized.train_test_split(test_size=0.01, seed=SEED)
train_ds  = split["train"]
eval_ds   = split["test"]
print(f"Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROB,
)

Train: 10,640,255 | Eval: 107,478


In [8]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Epochs & batch
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Optimizer
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",

    # Precision
    fp16=FP16,

    # Eval & Save
    eval_strategy=SAVE_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=SAVE_TOTAL,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    report_to="none",           # không cần wandb/tensorboard

    # Misc
    seed=SEED,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,9.889156,2.354783
2,9.230286,2.185318
3,8.953804,2.133271


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=249381, training_loss=9.991810103774954, metrics={'train_runtime': 38957.861, 'train_samples_per_second': 819.366, 'train_steps_per_second': 6.401, 'total_flos': 2.1974015501520064e+18, 'train_loss': 9.991810103774954, 'epoch': 3.0})

In [10]:
import math

# In perplexity cuối
metrics = trainer.evaluate()
ppl = math.exp(metrics["eval_loss"])
print(f"Eval Loss: {metrics['eval_loss']:.4f} | Perplexity: {ppl:.2f}")

# Lưu model + tokenizer
FINAL_DIR = f"{OUTPUT_DIR}/final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Model saved to {FINAL_DIR}")

Eval Loss: 2.1265 | Perplexity: 8.39


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /kaggle/working/phobert-continual/final


In [24]:
import shutil

# Đường dẫn thư mục cần nén
src_dir = "/kaggle/working/phobert-continual/checkpoint-249381"

# Tên file zip output
output_zip = "/kaggle/working/phobert-continual/checkpoint-249381"

# Nén
shutil.make_archive(output_zip, 'zip', src_dir)

print("Done! File saved at:", output_zip + ".zip")

Done! File saved at: /kaggle/working/phobert-continual/checkpoint-249381.zip


In [14]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
model = AutoModelForMaskedLM.from_pretrained(FINAL_DIR).to(device)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

mát
đẹp
mưa
nắng
lạnh


In [23]:

model.eval()

text = """Uầy em kia <mask> nhìn ngon thế"""  

inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]

probs = torch.softmax(logits[0, mask_token_index], dim=-1)
top_tokens = torch.topk(probs, 5, dim=-1).indices[0].tolist()

for token in top_tokens:
    print(tokenizer.decode([token]))

mặt
body
nhìn
sao
giờ
